# Day 2: QLoRA Multi-Adapter Training Runner
### **Routed Multi-Adapter LLM Serving System**

This notebook fine-tunes three specialized, rank-16 LoRA adapters on top of **`Qwen/Qwen2.5-1.5B-Instruct`**:
1. **`sql_lora`**: Natural language to SQL query generation (`data/sql_train.jsonl`)
2. **`json_lora`**: Semi-structured text to valid JSON extraction (`data/json_train.jsonl`)
3. **`code_lora`**: Functional Python code generation (`data/code_train.jsonl`)

**Compute Requirement:** 1x NVIDIA T4 GPU (16GB VRAM) via Google Colab (Free Tier) or Kaggle.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Install Dependencies
Installs Hugging Face libraries and bitsandbytes for 4-bit QLoRA.

In [ ]:
!pip install -q "transformers>=4.44.0" "peft>=0.12.0" "trl>=0.9.6" "bitsandbytes>=0.43.0" "accelerate>=0.33.0" "datasets>=2.18.0"

## 3. Clone / Setup Repository
If you have pushed this repository to GitHub, clone it below. Alternatively, upload your workspace folder into the Colab file browser.

In [ ]:
# If using GitHub (replace with your repo URL):
# !git clone https://github.com/<your-username>/routed-multi-adapter-serving.git
# %cd routed-multi-adapter-serving

# Verify that data/ and src/ exist
import os
assert os.path.exists('data/sql_train.jsonl'), 'data/sql_train.jsonl not found! Make sure repository files are uploaded.'
assert os.path.exists('src/train_loras.py'), 'src/train_loras.py not found!'
print('Repository structure verified successfully.')

## 4. Train SQL Adapter (`adapters/sql_lora`)
Trains for 2 epochs on 600 SQL examples (~5-8 mins on T4).

In [ ]:
!python -m src.train_loras --task sql --epochs 2 --batch-size 4 --grad-accum 2 --lr 2e-4

## 5. Train JSON Extraction Adapter (`adapters/json_lora`)
Trains for 2 epochs on 600 JSON extraction examples (~5-8 mins on T4).

In [ ]:
!python -m src.train_loras --task json --epochs 2 --batch-size 4 --grad-accum 2 --lr 2e-4

## 6. Train Code Generation Adapter (`adapters/code_lora`)
Trains for 2 epochs on 600 Python code examples (~5-8 mins on T4).

In [ ]:
!python -m src.train_loras --task code --epochs 2 --batch-size 4 --grad-accum 2 --lr 2e-4

## 7. Verify Adapter Checkpoints & Storage Overhead
Confirms that each adapter is saved in safetensors format and measures under 20 MB.

In [ ]:
import os

adapters = ['sql_lora', 'json_lora', 'code_lora']
print('=' * 60)
print(f'{"Adapter Name":<20} | {"Size on Disk":<15} | {"Status":<15}')
print('-' * 60)
for a in adapters:
    path = os.path.join('adapters', a)
    if os.path.exists(path):
        size_mb = sum(os.path.getsize(os.path.join(root, f)) for root, _, files in os.walk(path) for f in files) / (1024 * 1024)
        status = 'PASS (<20MB)' if size_mb < 20 else 'WARN (>20MB)'
        print(f'{a:<20} | {size_mb:.2f} MB        | {status}')
    else:
        print(f'{a:<20} | NOT FOUND       | FAILED')
print('=' * 60)

## 8. Export Adapters Archive
Compresses all 3 adapters into `adapters.zip` for local download or Google Drive backup.

In [ ]:
!zip -r adapters.zip adapters/

# If running in Google Colab, trigger direct browser download:
try:
    from google.colab import files
    files.download('adapters.zip')
    print('Download triggered for adapters.zip')
except ImportError:
    print('adapters.zip created successfully in the working directory.')